In [11]:
import pandas as pd
from data_profiling import ProfileReport
import numpy as np

# Partie 1 Profiling & audit

In [2]:
df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")
df.describe()

C:\Users\paulk\AppData\Local\Temp\ipykernel_53052\1620990980.py:1: DtypeWarning: Columns (12,18,19,20,21,22,24,25,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")


,siren_amenageur,nbre_pdc,puissance_nominale,consolidated_longitude,consolidated_latitude,consolidated_code_postal
count,1.362470e+05,227232.000000,227232.000000,227232.000000,227232.000000,134362.000000
mean,6.924200e+08,14.983180,102.212347,2.678738,46.733937,52603.766683
std,2.572975e+08,47.103571,579.594838,4.529209,4.034939,26780.472001
min,0.000000e+00,1.000000,0.000000,-149.905377,-44.996198,1000.000000
25%,5.243353e+08,2.000000,22.000000,0.774255,44.871519,31112.500000
50%,8.427185e+08,4.000000,22.000000,2.412432,47.340547,57160.000000
75%,8.951636e+08,9.000000,100.000000,4.836760,48.865021,76410.000000
max,9.921637e+08,505.000000,160000.000000,166.462000,61.520355,97418.000000


In [3]:
df.columns

Index(['nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'puissance_nominale', 'prise_type_ef', 'prise_type_2',
       'prise_type_combo_ccs', 'prise_type_chademo', 'prise_type_autre',
       'gratuit', 'paiement_acte', 'paiement_cb', 'paiement_autre',
       'tarification', 'condition_acces', 'reservation', 'horaires',
       'accessibilite_pmr', 'restriction_gabarit', 'station_deux_roues',
       'raccordement', 'num_pdl', 'date_mise_en_service', 'observations',
       'date_maj', 'cable_t2_attache', 'last_modified', 'datagouv_dataset_id',
       'datagouv_resource_id', 'datagouv_organization_or_owner', 'created_at',
       'consolidated_longitude', 'consolidated_latitude',
     

In [4]:
profile = ProfileReport(df, title="Profiling Report")

In [5]:
report_path = "rapport_data_profiling.html"
profile.to_file(report_path)

print(f"Rapport généré : {report_path}")

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 142 (\x8e) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 136 (\x88) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 143 (\x8f) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 143 (\x8f) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 142 (\x8e) missing from font(s) Arial.
  plt.savefig(
E

Rapport généré : rapport_data_profiling.html


In [6]:
def quality_score(df):
    total_cells = df.shape[0] * df.shape[1]

    missing = df.isna().sum().sum()
    completeness = 1 - missing / total_cells

    duplicates = df.duplicated().sum() / len(df)

    numeric_outliers = 0
    for col in df.select_dtypes("number"):
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        numeric_outliers += outliers

    outlier_ratio = numeric_outliers / total_cells

    score = (
        0.5 * completeness +
        0.3 * (1 - duplicates) +
        0.2 * (1 - outlier_ratio)
    )

    return round(score * 100, 2)

print(quality_score(df))

93.77


93.77% des données semblent de qualité en se basant sur le nombre de doublons, de valeurs aberrantes et de valeurs manquantes.

In [28]:
df["siren_amenageur"] = (
    df["siren_amenageur"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

Le type de "siren_amenageur" était un float, on l'a converti en str pour pouvoir effectuer du Regex dessus et checker sa validité 

In [29]:
audit_rows = []

# -------------------------
# 1. COMPLETUDE
# -------------------------
for col in df.columns:
    missing_rate = df[col].isna().mean()

    audit_rows.append({
        "dimension": "completude",
        "colonne": col,
        "pct_problemes": round(missing_rate * 100, 2),
        "gravite": "haute" if missing_rate > 0.3 else "moyenne" if missing_rate > 0.1 else "faible"
    })


# -------------------------
# 2. UNICITE (ID station)
# -------------------------
dup_rate = df.duplicated(subset=["id_station_itinerance"]).mean()

audit_rows.append({
    "dimension": "unicite",
    "colonne": "id_station_itinerance",
    "pct_problemes": round(dup_rate * 100, 2),
    "gravite": "haute" if dup_rate > 0.05 else "moyenne" if dup_rate > 0.01 else "faible"
})


# -------------------------
# 3. VALIDITE - SIREN
# -------------------------
siren_invalid = ~df["siren_amenageur"].astype(str).str.match(r"^\d{9}$")
siren_rate = siren_invalid.mean()

audit_rows.append({
    "dimension": "validite",
    "colonne": "siren_amenageur",
    "pct_problemes": round(siren_rate * 100, 2),
    "gravite": "haute" if siren_rate > 0.1 else "moyenne" if siren_rate > 0.02 else "faible"
})


# -------------------------
# 4. COHERENCE GEO
# -------------------------
geo_invalid = ~(
    df["consolidated_latitude"].between(41, 51) &
    df["consolidated_longitude"].between(-5, 10)
)

geo_rate = geo_invalid.mean()

audit_rows.append({
    "dimension": "coherence_geo",
    "colonne": "latitude_longitude",
    "pct_problemes": round(geo_rate * 100, 2),
    "gravite": "haute" if geo_rate > 0.2 else "moyenne" if geo_rate > 0.05 else "faible"
})


# -------------------------
# 5. VALIDITE PUISSANCE
# -------------------------
power_invalid = df["puissance_nominale"] <= 0
power_rate = power_invalid.mean()

audit_rows.append({
    "dimension": "validite",
    "colonne": "puissance_nominale",
    "pct_problemes": round(power_rate * 100, 2),
    "gravite": "haute" if power_rate > 0.1 else "moyenne" if power_rate > 0.02 else "faible"
})


# -------------------------
# RESULTAT FINAL
# -------------------------
audit_df = pd.DataFrame(audit_rows)
audit_df.sort_values("pct_problemes", ascending=False)

,dimension,colonne,pct_problemes,gravite
37,completude,observations,79.53,haute
27,completude,tarification,75.20,haute
52,unicite,id_station_itinerance,71.86,haute
35,completude,num_pdl,48.96,haute
39,completude,cable_t2_attache,46.97,haute
47,completude,consolidated_code_postal,40.87,haute
36,completude,date_mise_en_service,40.61,haute
53,validite,siren_amenageur,40.05,haute
34,completude,raccordement,34.90,haute
16,completude,id_pdc_local,34.62,haute


In [30]:
audit_df.groupby("dimension")["pct_problemes"].mean()

dimension
coherence_geo     0.530000
completude       11.496538
unicite          71.860000
validite         21.185000
Name: pct_problemes, dtype: float64

On voit que la catégorie qui pose le plus de problèmes est l'unicité avec 72% de données non uniques par colonnes en moyenne puis quelques problèmes de complétude, validité et de cohérence_geo

# Partie 2 Grain & doublons

In [7]:
df["num_pdl"].isna().sum()

np.int64(111260)

In [8]:
df.sample(10)

,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
2636,ELECTRIC 55 CHARGING,832489801.0,contact@e55c.com,ELECTRIC 55 CHARGING,contact@e55c.com,33975891501,ELECTRIC 55 CHARGING,FR55CP83210LAFP1PDGV,FR*55C*P83210*LAF*P1PDGV,PARKING DU GRAND VALLAT - LA FARLEDE,...,7a5acb37-32c9-48bf-a14f-cac23bb9aff0,plus-de-bornes,2023-08-21T10:10:50.570000+00:00,6.039670,43.160230,83210.0,La Farlède,True,True,False
54836,CityFMET,917546251.0,gestionfournisseurs@oriosbyspie.com,SPIE CityNetworks,assistance-technique@e-vadea.fr,tel:+33-9-70-83-03-87,e-Vadea,FREVAP22353A,FR*EVA*P22353*A,e-Vadea - Trégastel - Anne,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,-3.511590,48.827034,22730.0,Trégastel,True,True,False
89684,IZIVIA FAST,951478437.0,lucas.malacarne@izivia.com,IZIVIA,sav@izivia.com,+33(0)972668001,IZIVIA FAST,FRIZFPFAST481,FRSODPFAST4811,IZIVIA FAST - McDonald's - MENDE,...,7696ac9e-2789-4eb6-9091-8a9b44cf0721,izivia,2023-11-30T09:39:27.928000+00:00,3.478046,44.514801,48000.0,Mende,True,True,False
14971,Atlante,911482628.0,digital@atlante.energy,Atlante France,support.france@atlante.energy,tel:+33-1-83-75-07-25,Atlante - Montpellier - Centre commercial Odys...,FRATLPPZ331FRODYSSE000002,NaN,Atlante - Montpellier - Centre commercial Odys...,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,3.920019,43.602627,NaN,Montpellier,True,True,False
6771,Allego,NaN,info@allego.eu,Allego,info@allego.eu,+33 187 40 66 57,Allego - Burger King Cavaillon,FRALLEGO6001362,FRALLEGO6001362,Allego - Burger King Cavaillon,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,5.055768,43.829612,NaN,NaN,False,False,False
198858,Tesla,524335262.0,charging-france@tesla.com,Tesla,charging-france@tesla.com,tel:+33-9-70-73-08-50,Tesla,FRTSLP30513,fe2a3283-6bc4-4fdb-936c-01e6a5e5f0f8,"Nîmes, France - Carré Sud",...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,4.364058,43.807830,NaN,Nîmes,True,True,False
147456,Plenitude On The Road,NaN,support@service.emob.eniplenitude.com,Plenitude On The Road,support@service.emob.eniplenitude.com,+(33)-(1)-73049601,Eni Station - RN 7 La Tuilerie,FRPLNEH001083,FRPLNEH001083,Eni Station - RN 7 La Tuilerie,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,6.702163,43.442411,NaN,NaN,False,False,False
75922,TotalEnergies Marketing France,531680445.0,supervision-ev.france@totalenergies.com,TotalEnergies Marketing France,supervision-ev.france@totalenergies.com,tel:+33-1-85-16-94-02,TotalEnergies Charge Rapide,FRHPCPNF078556TOTEM,FR*HPC*PNF078556*TOTEM,RELAIS LA DENTELLE D'ALENCON,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,0.125336,48.456669,61250.0,Valframbert,True,True,False
212623,ENGIE Vianeo,909073363.0,support.vianeo@engie.com,ENGIE Vianeo,support.vianeo@engie.com,tel:+33-9-69-37-60-09,ENGIE Vianeo,FRVIAP121033,121033,ENGIE Vianeo - Hôtel Campanile Besançon Châteauf.,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,5.943245,47.216541,25000.0,Besançon,True,True,False
193905,TotalEnergies Charging Services,844192443.0,supervision-ev.france@totalenergies.com,TotalEnergies Charging Services,supervision-ev.france@totalenergies.com,tel:+33-4-83-56-80-09,TotalEnergies - Metpark,FRTCBPMETAMP,FR*TCB*PMETAMP,METPARK | TotalEnergies | Parking Amplitude,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,-0.554840,44.839208,NaN,Bordeaux,True,True,False


In [27]:
df['siren_amenageur']

0                 NaN
1                 NaN
2                 NaN
3                 NaN
4                 NaN
             ...     
227227    895193019.0
227228    508013281.0
227229    444811970.0
227230    444811970.0
227231    444811970.0
Name: siren_amenageur, Length: 227232, dtype: float64

In [9]:
columnes_a_verifier = [
    col for col in df.columns
    if "id_pdc" in col.lower()
    or "itiner" in col.lower()
    or "station" in col.lower()
    or "date" in col.lower()
    or "operateur" in col.lower()
]

print("Colonnes candidates:")
print(columnes_a_verifier)

col_id = "id_pdc_itinerance"
print("\nNombre de lignes:", len(df))
print("Valeurs distinctes de id_pdc_itinerance:", df[col_id].nunique(dropna=False))
print("Lignes en doublon sur id_pdc_itinerance:", df.duplicated(subset=[col_id]).sum())

freq_id = df[col_id].value_counts(dropna=False)
print("\nTop 10 des valeurs les plus fréquentes:")
print(freq_id.head(10))

if len(columnes_a_verifier) > 1:
    print("\nTest de granularité sur plusieurs colonnes:")
    for taille in range(2, min(5, len(columnes_a_verifier)) + 1):
        subset = columnes_a_verifier[:taille]
        doublons = df.duplicated(subset=subset).sum()
        print(f"{subset} -> doublons: {doublons}")

Colonnes candidates:
['nom_operateur', 'contact_operateur', 'telephone_operateur', 'id_station_itinerance', 'id_station_local', 'nom_station', 'implantation_station', 'adresse_station', 'id_pdc_itinerance', 'id_pdc_local', 'station_deux_roues', 'date_mise_en_service', 'date_maj', 'consolidated_longitude', 'consolidated_latitude', 'consolidated_code_postal', 'consolidated_commune', 'consolidated_is_lon_lat_correct', 'consolidated_is_code_insee_verified', 'consolidated_is_code_insee_modified']

Nombre de lignes: 227232
Valeurs distinctes de id_pdc_itinerance: 160138
Lignes en doublon sur id_pdc_itinerance: 67094

Top 10 des valeurs les plus fréquentes:
id_pdc_itinerance
Non concerné       118
FRLIBE3126085        4
FRCG0E002113         3
FRCG0E002114         3
FRCG0E002115         3
FRCG0E002116         3
FRCG0E001353         3
FRALLEGO9004771      3
FRALLEGO9004772      3
FRALLEGO9004773      3
Name: count, dtype: int64

Test de granularité sur plusieurs colonnes:
['nom_operateur', 'con

In [10]:
id_exemple = freq_id.index[1]
exemple = df.loc[df[col_id] == id_exemple, [
    col for col in [
        "id_station_itinerance",
        "id_pdc_itinerance",
        "id_pdc_local",
        "nom_station",
        "nom_operateur",
        "date_mise_en_service",
        "date_maj",
        "consolidated_commune",
        "consolidated_code_postal",
    ] if col in df.columns
]]
print(f"Exemple pour {id_exemple}:")
display(exemple)

Exemple pour FRLIBE3126085:


,id_station_itinerance,id_pdc_itinerance,id_pdc_local,nom_station,nom_operateur,date_mise_en_service,date_maj,consolidated_commune,consolidated_code_postal
105425,FRLIBE3126085,FRLIBE3126085,frlibe3126085,Résidence Carouge,Bornevo,2023-06-23,2023-10-21,Brétigny-sur-Orge,91220.0
105426,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,NaN,2023-11-10,2023-11-10,NaN,NaN
105427,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,Eoliberty,2023-10-11,2023-10-11,Brétigny-sur-Orge,91220.0
105428,FRLIBE3126085,FRLIBE3126085,NaN,Residence Carouge,Eoliberty,2023-10-11,2023-10-11,Brétigny-sur-Orge,91220.0


id_pdc_itinerance n'est pas une clef car une ligne représente un snapshot de l'évolution de la mission d'un opérateur. Donc une itinérance peut apparaitre pour plusieurs dates différentes, nom d'opérateur, point de charge local.

Ce qui nous interesse nous, c'est d'avoir une ligne par point de livraison x point de charge

In [32]:
dup_rate = df.duplicated(subset=["id_pdc_itinerance", "num_pdl"]).mean()

print("Taux de doublons PDC×PDL :", round(dup_rate * 100, 2), "%")

Taux de doublons PDC×PDL : 12.43 %


On a tout de même 12.43% de doublons c'est à dire de lignes qui ont le même couple (PDL,PDC)
Pour être sûr de garder un dataset propre, on ne va garder dans les doublons que ceux de la dernière date de mise à jour

In [33]:
df["date_maj"] = pd.to_datetime(df["date_maj"], errors="coerce")

In [34]:
df_sorted = df.sort_values("date_maj", ascending=False)

In [35]:
df_clean = df_sorted.drop_duplicates(
    subset=["id_pdc_itinerance", "num_pdl"],
    keep="first"
)

In [39]:
nb_bornes_clean = df_clean["id_pdc_itinerance"].nunique()
print("Nombre de bornes après:", nb_bornes_clean)


Nombre de bornes après: 160138


On a 160 138 bornes